# Subir video, extraer WAV y subir ambos a Google Drive (sin MongoDB)

Este notebook:
1. Sube un archivo de video desde tu equipo.
2. Extrae audio WAV con ffmpeg (44.1 kHz, estéreo, PCM16).
3. Sube el video y el audio a Google Drive manteniendo el nombre original (el audio será el mismo nombre pero con .wav).

Ejecuta las celdas en orden y luego llama a `upload_and_process_video()`.

In [ ]:
# Instalar dependencias necesarias (sin MongoDB)
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os
import time
import subprocess
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import files
import shutil
from pathlib import Path

# ==========================
# Configuración de credenciales de Google Drive
# ==========================
CLIENT_ID = '448213662107-7fuh61iinmeg78df9epa6ve54qm7mn2o.apps.googleusercontent.com'
CLIENT_SECRET = 'YOUR_GOOGLE_OAUTH_CLIENT_SECRET'
REFRESH_TOKEN = 'YOUR_GOOGLE_OAUTH_REFRESH_TOKEN'

# IDs de carpetas en Drive
FOLDER_VIDEOS = '195TbCi27MxDtzV9RATxQ2bBHwsSDUmzb'  # Carpeta de videos
FOLDER_AUDIOS = '1fjmlPFGe8QEXoPjcAZFMFd6AkErkFVCI'  # Carpeta de audios

def _build_drive_service():
    creds = Credentials(
        None,
        refresh_token=REFRESH_TOKEN,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        token_uri='https://oauth2.googleapis.com/token'
    )
    return build('drive', 'v3', credentials=creds)

def _extract_wav(input_path: str, output_path: str):
    subprocess.run([
        'ffmpeg', '-y', '-i', input_path,
        '-vn', '-acodec', 'pcm_s16le',
        '-ar', '44100', '-ac', '2',
        output_path
    ], check=True)

def _upload_to_drive(service, local_path: str, drive_name: str, parent_folder_id: str):
    media = MediaFileUpload(local_path)
    metadata = {'name': drive_name, 'parents': [parent_folder_id]}
    file_obj = service.files().create(body=metadata, media_body=media, fields='id,webContentLink').execute()
    # Cerrar file descriptors si existen
    if hasattr(media, '_fd') and getattr(media._fd, 'closed', True) is False:
        media._fd.close()
    return file_obj

def upload_and_process_video():
    """Sube un video, extrae WAV y sube ambos a Drive manteniendo el nombre original."""
    print('Selecciona el archivo de video para subir:')
    uploaded = files.upload()
    if not uploaded:
        print('No se seleccionó ningún archivo')
        return None

    original_name = list(uploaded.keys())[0]
    base = str(Path(original_name).stem)
    wav_name = base + '.wav'

    os.makedirs('temp', exist_ok=True)
    video_path = os.path.join('temp', original_name)
    # Mover el archivo subido a temp conservando el nombre
    shutil.move(original_name, video_path)
    print(f'Archivo guardado como: {video_path}')

    try:
        print('Extrayendo audio (WAV)...')
        wav_path = os.path.join('temp', wav_name)
        _extract_wav(video_path, wav_path)
        print(f'Audio extraído: {wav_path}')

        print('Autenticando con Google Drive...')
        service = _build_drive_service()

        print('Subiendo video a Google Drive...')
        video_upload = _upload_to_drive(service, video_path, original_name, FOLDER_VIDEOS)
        print(f"Video subido - ID: {video_upload.get('id')}")

        print('Subiendo audio a Google Drive...')
        audio_upload = _upload_to_drive(service, wav_path, wav_name, FOLDER_AUDIOS)
        print(f"Audio subido - ID: {audio_upload.get('id')}")

        # Limpieza local
        time.sleep(0.3)
        for p in (video_path, wav_path):
            try:
                if os.path.exists(p):
                    os.remove(p)
            except Exception:
                pass

        result = {
            'video': {
                'name': original_name,
                'file_id': video_upload.get('id'),
                'download_url': video_upload.get('webContentLink')
            },
            'audio': {
                'name': wav_name,
                'file_id': audio_upload.get('id'),
                'download_url': audio_upload.get('webContentLink')
            }
        }

        print('\n=== PROCESO COMPLETADO ===')
        print(f"Video URL: {result['video']['download_url']}")
        print(f"Audio URL: {result['audio']['download_url']}")
        return result

    except subprocess.CalledProcessError as ffmpeg_error:
        print(f'Error con ffmpeg: {ffmpeg_error}')
        return None
    except Exception as e:
        print(f'Error general: {e}')
        return None

print('=== Cargado. Llama a upload_and_process_video() para iniciar ===')

In [ ]:
# Ejecuta esta celda para iniciar el flujo
result = upload_and_process_video()
result